# 12 — Episode Sequential Behavior Analysis

This notebook analyzes the **collection of 270 scored episodes** (90 per run) from two complementary angles:

1. **Final Itinerary Composition** — what each run actually booked: flights, hotels, activities, and costs.
2. **Sequential Tool-Use Behavior** — not just counts but the *ordered structure* of tool sequences, revealing how each run navigates the planning task.

**Data sources**
- `outputs/consolidated_episode_evals/eval_results-001.csv` — eval metadata with `episode_id → run_name` mapping
- `outputs/episodes/ep_*.json` — full step-level trajectory data

**Key fields extracted from episode JSONs (no free-text parsing)**
- `trajectory.steps[i].action.tool_name`
- `trajectory.steps[i].observation.success`
- `trajectory.steps[i].itinerary_snapshot` (null or not)
- `final_itinerary.days[d].transport_segments`, `.accommodation`, `.activities`

## 0. Setup

In [ ]:
import json
import math
import warnings
from collections import Counter, defaultdict
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110

In [ ]:
REPO_ROOT = Path("..")
EVAL_CSV   = REPO_ROOT / "outputs" / "consolidated_episode_evals" / "eval_results-001.csv"
EPISODES_DIR = REPO_ROOT / "outputs" / "episodes"

assert EVAL_CSV.exists(),      f"CSV not found: {EVAL_CSV}"
assert EPISODES_DIR.exists(),  f"Episodes dir not found: {EPISODES_DIR}"

# Palette — one colour per run_name, assigned after loading
RUN_PALETTE: dict[str, str] = {}

## 1. Data Loading

In [ ]:
# Load CSV — results_loader writes MultiIndex columns (two header rows)
try:
    meta_df = pd.read_csv(EVAL_CSV, header=[0, 1], low_memory=False)
    # Confirm it looks like multi-level: top-level groups should be short tokens
    if not any(len(str(c[0])) < 20 for c in meta_df.columns[:5]):
        raise ValueError("Does not look like a two-level header")
    print(f"Loaded CSV with multi-level columns: {meta_df.shape[0]} rows, top-level groups: {sorted(set(c[0] for c in meta_df.columns))}")
    
    def col(group: str, field: str) -> pd.Series:
        """Access column by (group, field) tuple."""
        return meta_df[(group, field)]

except Exception:
    # Fallback: flat column names like 'ids.episode_id'
    meta_df = pd.read_csv(EVAL_CSV, low_memory=False)
    print(f"Loaded CSV with flat columns: {meta_df.shape[0]} rows")
    
    def col(group: str, field: str) -> pd.Series:
        candidates = [f"{group}.{field}", f"{group}_{field}", field]
        for c in candidates:
            if c in meta_df.columns:
                return meta_df[c]
        raise KeyError(f"Column not found for ({group}, {field}). Available: {list(meta_df.columns[:20])}")

# Build episode_id → run metadata mapping
ep_meta: dict[str, dict] = {}
for _, row in meta_df.iterrows():
    eid = str(col("ids", "episode_id").iloc[_] if False else row[("ids", "episode_id") if isinstance(meta_df.columns, pd.MultiIndex) else "ids.episode_id"])
    ep_meta[eid] = {
        "run_name":          str(row[("run", "run_name")    if isinstance(meta_df.columns, pd.MultiIndex) else "run.run_name"]),
        "agent_mode":        str(row[("config", "agent_mode") if isinstance(meta_df.columns, pd.MultiIndex) else "config.agent_mode"]),
        "total_steps":       row[("episode", "total_steps")  if isinstance(meta_df.columns, pd.MultiIndex) else "episode.total_steps"],
        "success":           row[("episode", "success")      if isinstance(meta_df.columns, pd.MultiIndex) else "episode.success"],
        "termination_reason":row[("episode", "termination_reason") if isinstance(meta_df.columns, pd.MultiIndex) else "episode.termination_reason"],
    }

print(f"Episode metadata rows: {len(ep_meta)}")
run_names = sorted(set(v["run_name"] for v in ep_meta.values()))
print(f"Run names: {run_names}")

# Assign palette
palette_colors = sns.color_palette("tab10", len(run_names))
RUN_PALETTE = {name: palette_colors[i] for i, name in enumerate(run_names)}

In [ ]:
# Helper: flatten multi-level column access without assumptions about format
def _get(row, group: str, field: str):
    key_multi = (group, field)
    key_flat  = f"{group}.{field}"
    if isinstance(meta_df.columns, pd.MultiIndex):
        return row[key_multi]
    return row[key_flat]


def _load_json(path: Path) -> dict | None:
    try:
        with path.open(encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


# Load all episode JSONs that have metadata (scored episodes only)
all_episodes: list[dict] = []
missing = 0

for ep_file in sorted(EPISODES_DIR.glob("ep_*.json")):
    # Filename: ep_{uuid}.json → extract uuid
    eid = ep_file.stem.removeprefix("ep_")
    if eid not in ep_meta:
        continue  # unscored episode
    data = _load_json(ep_file)
    if data is None:
        missing += 1
        continue
    data["_meta"] = ep_meta[eid]
    all_episodes.append(data)

print(f"Loaded {len(all_episodes)} episodes ({missing} failed to parse)")
print(f"By run_name: { {rn: sum(1 for e in all_episodes if e['_meta']['run_name'] == rn) for rn in run_names} }")

---
## 2. Final Itinerary Composition

For each episode, extract from `final_itinerary.days[]`:
- **n_transport_segments** — total booked transport legs; broken down by mode (flight, train, etc.)
- **n_hotel_stays** — number of `ItineraryDay` objects with a non-null accommodation
- **n_activities** — total `ActivityBooking` objects across all days
- **total_cost_usd** — `Itinerary.total_cost_usd` (auto-computed from day sums)
- **cost breakdown** — by transport, hotel, activities

In [ ]:
def extract_itinerary_stats(ep: dict) -> dict:
    itin = ep.get("final_itinerary")
    base = {
        "episode_id": ep["episode_id"],
        "run_name": ep["_meta"]["run_name"],
        "agent_mode": ep["_meta"]["agent_mode"],
        "has_itinerary": itin is not None,
        "is_complete": False,
        "n_days": 0,
        "n_transport_segments": 0,
        "n_flights": 0,
        "n_other_transport": 0,
        "n_hotel_stays": 0,
        "n_activities": 0,
        "total_cost_usd": 0.0,
        "flight_cost_usd": 0.0,
        "hotel_cost_usd": 0.0,
        "activity_cost_usd": 0.0,
    }
    if itin is None:
        return base
    base["is_complete"] = bool(itin.get("is_complete", False))
    base["total_cost_usd"] = float(itin.get("total_cost_usd", 0.0))
    days = itin.get("days", [])
    base["n_days"] = len(days)
    for day in days:
        for seg in day.get("transport_segments", []):
            base["n_transport_segments"] += 1
            cost = float(seg.get("cost_usd", 0.0))
            if seg.get("mode", "").lower() == "flight":
                base["n_flights"] += 1
                base["flight_cost_usd"] += cost
            else:
                base["n_other_transport"] += 1
        acc = day.get("accommodation")
        if acc:
            base["n_hotel_stays"] += 1
            base["hotel_cost_usd"] += float(acc.get("total_cost_usd", 0.0))
        for act in day.get("activities", []):
            base["n_activities"] += 1
            base["activity_cost_usd"] += float(act.get("cost_usd", 0.0))
    return base

itin_rows = [extract_itinerary_stats(ep) for ep in all_episodes]
itin_df = pd.DataFrame(itin_rows)
print(f"Shape: {itin_df.shape}")
print(f"Episodes with itinerary: {itin_df['has_itinerary'].sum()} / {len(itin_df)}")
itin_df.groupby("run_name")[["has_itinerary", "is_complete", "n_flights", "n_hotel_stays", "n_activities", "total_cost_usd"]].agg(["mean", "std"]).round(2)

In [ ]:
# 2a. Mean booking counts by run_name (only episodes with a non-null itinerary)
has_itin = itin_df[itin_df["has_itinerary"]].copy()

count_cols = ["n_flights", "n_hotel_stays", "n_activities", "n_other_transport"]
count_labels = ["Flights", "Hotel nights", "Activities", "Other transport"]

means = has_itin.groupby("run_name")[count_cols].mean()
sems  = has_itin.groupby("run_name")[count_cols].sem()

x = np.arange(len(count_cols))
width = 0.8 / len(run_names)

fig, ax = plt.subplots(figsize=(10, 4))
for i, rn in enumerate(run_names):
    if rn not in means.index:
        continue
    ax.bar(x + i * width, means.loc[rn, count_cols], width,
           yerr=sems.loc[rn, count_cols], capsize=3,
           label=rn, color=RUN_PALETTE[rn], alpha=0.85)

ax.set_xticks(x + width * (len(run_names) - 1) / 2)
ax.set_xticklabels(count_labels)
ax.set_ylabel("Mean count (episodes with itinerary)")
ax.set_title("2a. Mean Booking Counts by Run (±1 SEM, episodes with itinerary only)")
ax.legend(title="Run", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# 2b. Total cost distribution (box plot) — all episodes (zeros for no itinerary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: all episodes (including zeros)
itin_df.boxplot(column="total_cost_usd", by="run_name", ax=axes[0],
                patch_artist=True,
                boxprops=dict(color="steelblue"),
                medianprops=dict(color="firebrick", linewidth=2))
axes[0].set_title("Total cost (all episodes, zeros included)")
axes[0].set_xlabel("")
axes[0].set_ylabel("Total cost (USD)")

# Right: non-zero itineraries only
has_itin.boxplot(column="total_cost_usd", by="run_name", ax=axes[1],
                 patch_artist=True,
                 boxprops=dict(color="darkorange"),
                 medianprops=dict(color="firebrick", linewidth=2))
axes[1].set_title("Total cost (episodes with itinerary)")
axes[1].set_xlabel("")
axes[1].set_ylabel("Total cost (USD)")

fig.suptitle("2b. Itinerary Total Cost Distribution by Run")
plt.tight_layout()
plt.show()

In [ ]:
# 2c. Stacked cost breakdown proportions by run_name
cost_cols = ["flight_cost_usd", "hotel_cost_usd", "activity_cost_usd"]
cost_labels = ["Flights", "Hotels", "Activities"]

breakdown = has_itin.groupby("run_name")[cost_cols].mean()
# Normalize to proportions
breakdown_pct = breakdown.div(breakdown.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

breakdown.plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"],
               edgecolor="white", stacked=True)
axes[0].set_title("Mean absolute cost breakdown")
axes[0].set_xlabel("")
axes[0].set_ylabel("Cost (USD)")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(cost_labels, title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")

breakdown_pct.plot(kind="bar", ax=axes[1], color=["#4C72B0", "#DD8452", "#55A868"],
                   edgecolor="white", stacked=True)
axes[1].set_title("Cost share (%) by category")
axes[1].set_xlabel("")
axes[1].set_ylabel("Share (%)")
axes[1].tick_params(axis="x", rotation=25)
axes[1].legend(cost_labels, title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")

fig.suptitle("2c. Cost Breakdown by Run")
plt.tight_layout()
plt.show()

In [ ]:
# 2d. Total cost vs total steps — coloured by run_name
meta_df_flat = meta_df.copy()
# Merge itinerary stats with total_steps from meta
plot_df = itin_df.merge(
    meta_df_flat[[c for c in meta_df_flat.columns if (isinstance(c, tuple) and c == ("ids","episode_id")) or c == "ids.episode_id"]][:0],  # just schema check
    left_on="episode_id", right_on=None, how="left"
).copy()
plot_df["total_steps"] = plot_df["episode_id"].map(lambda eid: ep_meta.get(eid, {}).get("total_steps", np.nan))

fig, ax = plt.subplots(figsize=(9, 5))
for rn in run_names:
    sub = plot_df[plot_df["run_name"] == rn]
    ax.scatter(sub["total_steps"], sub["total_cost_usd"],
               label=rn, color=RUN_PALETTE[rn], alpha=0.5, s=25, edgecolors="none")

ax.set_xlabel("Total steps in episode")
ax.set_ylabel("Final itinerary total cost (USD)")
ax.set_title("2d. Total Cost vs Episode Length (0 = no itinerary)")
ax.legend(title="Run", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

---
## 3. Sequential Tool-Use Behavior

Build a flat table of all (episode, step) records, then derive sequential statistics.

Fields extracted per step:
- `step_index`
- `action.tool_name`
- `observation.success`
- `itinerary_snapshot` — null vs non-null (has a booking appeared yet?)

In [ ]:
step_records: list[dict] = []

for ep in all_episodes:
    eid      = ep["episode_id"]
    run_name = ep["_meta"]["run_name"]
    agent_mode = ep["_meta"]["agent_mode"]
    traj     = ep.get("trajectory", {})
    steps    = traj.get("steps", []) if isinstance(traj, dict) else []
    n_steps  = len(steps)

    for step in steps:
        action = step.get("action") or {}
        obs    = step.get("observation") or {}
        step_records.append({
            "episode_id":       eid,
            "run_name":         run_name,
            "agent_mode":       agent_mode,
            "step_index":       int(step.get("step_index", -1)),
            "n_steps_total":    n_steps,
            "tool_name":        str(action.get("tool_name", "UNKNOWN")),
            "success":          bool(obs.get("success", False)),
            "has_booking":      step.get("itinerary_snapshot") is not None,
        })

steps_df = pd.DataFrame(step_records)

# Relative step position 0.0–1.0 (within each episode)
steps_df["rel_pos"] = steps_df["step_index"] / steps_df["n_steps_total"].clip(lower=1)
steps_df["rel_decile"] = (steps_df["rel_pos"] * 10).clip(upper=9.99).astype(int)

print(f"Total step records: {len(steps_df):,}")
print(f"Distinct tool names seen: {sorted(steps_df['tool_name'].unique())}")
steps_df.groupby("run_name")[["step_index"]].agg(["count", "max"]).rename(columns={"count": "n_steps", "max": "max_step"})

### 3.1 Tool Inventory & Sequence Lengths

In [ ]:
# Tool call frequency by run_name (normalised within run)
tool_freq = (
    steps_df.groupby(["run_name", "tool_name"])
    .size()
    .reset_index(name="count")
)
total_by_run = steps_df.groupby("run_name").size().rename("total")
tool_freq = tool_freq.merge(total_by_run, on="run_name")
tool_freq["pct"] = tool_freq["count"] / tool_freq["total"] * 100

# Pivot for heatmap
tool_pivot = tool_freq.pivot(index="run_name", columns="tool_name", values="pct").fillna(0)

fig, ax = plt.subplots(figsize=(14, max(3, len(run_names) * 0.8)))
sns.heatmap(tool_pivot, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.4,
            cbar_kws={"label": "% of all steps"}, ax=ax)
ax.set_title("3.1 Tool Call Frequency (% of episode steps) by Run")
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

print("\nMost-called tool per run:")
print(tool_freq.sort_values("count", ascending=False).groupby("run_name").first()[["tool_name", "pct"]].rename(columns={"tool_name": "top_tool", "pct": "%"}))

In [ ]:
# Distribution of episode lengths by run_name
ep_lengths = steps_df.groupby(["run_name", "episode_id"])["step_index"].max().reset_index()
ep_lengths.columns = ["run_name", "episode_id", "max_step"]

fig, ax = plt.subplots(figsize=(9, 4))
for rn in run_names:
    sub = ep_lengths[ep_lengths["run_name"] == rn]["max_step"]
    ax.hist(sub, bins=20, alpha=0.55, label=rn, color=RUN_PALETTE[rn], density=False)

ax.set_xlabel("Max step index (= total steps − 1)")
ax.set_ylabel("Number of episodes")
ax.set_title("3.1 Episode Length Distribution by Run")
ax.legend(title="Run")
plt.tight_layout()
plt.show()

### 3.2 Markov Chain: P(next tool | current tool)

Count consecutive tool transitions across all episodes within each run, then normalise rows to probabilities. Self-loops indicate repeated calls to the same tool.

**Interpretation guide:** A high diagonal value (self-loop) signals the agent is stuck calling the same tool repeatedly. Off-diagonal patterns reveal standard planning sequences (e.g. `search_flights → select_flight → search_hotels`).

In [ ]:
def build_markov_matrix(df: pd.DataFrame, run: str) -> pd.DataFrame:
    """Return row-normalised transition matrix for one run."""
    sub = df[df["run_name"] == run].sort_values(["episode_id", "step_index"])
    counts: dict[tuple[str, str], int] = defaultdict(int)
    for ep_id, grp in sub.groupby("episode_id"):
        tools = grp["tool_name"].tolist()
        for a, b in zip(tools, tools[1:]):
            counts[(a, b)] += 1
    if not counts:
        return pd.DataFrame()
    all_tools = sorted(set(t for pair in counts for t in pair))
    mat = pd.DataFrame(0.0, index=all_tools, columns=all_tools)
    for (a, b), c in counts.items():
        mat.loc[a, b] += c
    # Row-normalise
    row_sums = mat.sum(axis=1)
    mat = mat.div(row_sums.where(row_sums > 0, 1.0), axis=0)
    return mat

markov_matrices = {rn: build_markov_matrix(steps_df, rn) for rn in run_names}

n_runs = len(run_names)
fig, axes = plt.subplots(1, n_runs, figsize=(7 * n_runs, max(6, len(run_names) * 1.5 + 4)))
if n_runs == 1:
    axes = [axes]

for ax, rn in zip(axes, run_names):
    mat = markov_matrices[rn]
    if mat.empty:
        ax.set_title(f"{rn}\n(no data)")
        continue
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                linewidths=0.3, cbar=True, ax=ax,
                cbar_kws={"shrink": 0.6, "label": "P(next | current)"})
    ax.set_title(f"3.2 Markov Transitions\n{rn}", fontsize=10)
    ax.set_xlabel("Next tool")
    ax.set_ylabel("Current tool")
    ax.tick_params(axis="x", rotation=40, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

plt.suptitle("Tool Transition Probabilities P(next | current)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Self-loop probability summary: P(next == current) for each tool and run
print("Self-loop probabilities (P(same tool again | current tool)):")
for rn in run_names:
    mat = markov_matrices[rn]
    if mat.empty:
        continue
    self_loops = pd.Series({t: mat.loc[t, t] for t in mat.index if t in mat.columns}, name=rn)
    top = self_loops.sort_values(ascending=False).head(5)
    print(f"\n  {rn}:")
    for tool, p in top.items():
        print(f"    {tool:35s}  P(repeat) = {p:.3f}")

### 3.3 Tool Bigram & Trigram Patterns

The most common two- and three-tool subsequences reveal stereotyped planning sub-routines. For example, a high-frequency `(get_available_routes, get_available_routes)` bigram suggests the agent re-discovers city IDs it already has.

In [ ]:
def ngrams(seq: list, n: int):
    return [tuple(seq[i:i+n]) for i in range(len(seq) - n + 1)]

def top_ngrams(df: pd.DataFrame, run: str, n: int, k: int = 10) -> pd.Series:
    sub = df[df["run_name"] == run].sort_values(["episode_id", "step_index"])
    counter: Counter = Counter()
    for _, grp in sub.groupby("episode_id"):
        counter.update(ngrams(grp["tool_name"].tolist(), n))
    top = counter.most_common(k)
    labels = [" → ".join(t) for t, _ in top]
    counts = [c for _, c in top]
    return pd.Series(counts, index=labels, name=run)

for gram_size, gram_label in [(2, "Bigrams"), (3, "Trigrams")]:
    fig, axes = plt.subplots(1, n_runs, figsize=(8 * n_runs, 5), sharey=False)
    if n_runs == 1:
        axes = [axes]
    for ax, rn in zip(axes, run_names):
        s = top_ngrams(steps_df, rn, gram_size, k=10)
        if s.empty:
            ax.set_title(f"{rn}\n(no data)")
            continue
        s.sort_values().plot(kind="barh", ax=ax, color=RUN_PALETTE[rn], edgecolor="white")
        ax.set_title(f"{rn}")
        ax.set_xlabel("Count")
        ax.tick_params(axis="y", labelsize=7.5)
    fig.suptitle(f"3.3 Top-10 Tool {gram_label} by Run", fontsize=13)
    plt.tight_layout()
    plt.show()

### 3.4 Error Cascade Analysis

After a failed tool call, does the agent:
- **Recover** — call a *different* tool next?
- **Retry** — call the *same* tool again?
- **Cascade** — fail again regardless?

We compute P(next outcome | previous outcome) and the retry rate after failures.

In [ ]:
cascade_stats: list[dict] = []

for rn in run_names:
    sub = steps_df[steps_df["run_name"] == rn].sort_values(["episode_id", "step_index"])
    ss_count = sf_count = fs_count = ff_count = 0
    retry_after_fail = same_after_fail = 0

    for _, grp in sub.groupby("episode_id"):
        succ = grp["success"].tolist()
        tools = grp["tool_name"].tolist()
        for i in range(len(succ) - 1):
            a, b = succ[i], succ[i + 1]
            if a and b:   ss_count += 1
            elif a and not b: sf_count += 1
            elif not a and b: fs_count += 1
            else:         ff_count += 1
            if not a:  # after failure
                retry_after_fail += 1
                if tools[i] == tools[i + 1]:
                    same_after_fail += 1

    total = ss_count + sf_count + fs_count + ff_count
    cascade_stats.append({
        "run_name": rn,
        "P(s→s)": ss_count / max(ss_count + sf_count, 1),
        "P(s→f)": sf_count / max(ss_count + sf_count, 1),
        "P(f→s)": fs_count / max(fs_count + ff_count, 1),
        "P(f→f)": ff_count / max(fs_count + ff_count, 1),
        "retry_rate": same_after_fail / max(retry_after_fail, 1),
        "n_post_fail_steps": retry_after_fail,
    })

cascade_df = pd.DataFrame(cascade_stats).set_index("run_name")
print(cascade_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Outcome-transition heatmap
transition_cols = ["P(s→s)", "P(s→f)", "P(f→s)", "P(f→f)"]
hm_data = cascade_df[transition_cols]
sns.heatmap(hm_data, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=0.4, ax=axes[0], cbar_kws={"label": "Probability"})
axes[0].set_title("3.4a Outcome Transition Probabilities\n(s=success, f=failure)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

# Retry rate bar
retry_series = cascade_df["retry_rate"]
bars = axes[1].bar(retry_series.index, retry_series.values,
                   color=[RUN_PALETTE[rn] for rn in retry_series.index],
                   edgecolor="white", alpha=0.85)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Retry rate (same tool after failure)")
axes[1].set_title("3.4b Retry Rate After a Failed Call")
axes[1].tick_params(axis="x", rotation=20)
for bar, val in zip(bars, retry_series.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.2f}",
                 ha="center", va="bottom", fontsize=9)

plt.suptitle("3.4 Error Cascade Analysis")
plt.tight_layout()
plt.show()

### 3.5 Planning Momentum — When Does Booking Begin?

`itinerary_snapshot` turns non-null the moment the agent produces any booking. The step at which this first occurs measures how quickly the agent commits to an actionable plan — a proxy for planning efficiency.

In [ ]:
# For each episode: first step with has_booking == True
first_booking = (
    steps_df[steps_df["has_booking"]]
    .groupby(["episode_id", "run_name"])["step_index"]
    .min()
    .reset_index()
    .rename(columns={"step_index": "first_booking_step"})
)

all_eps = steps_df[["episode_id", "run_name"]].drop_duplicates()
momentum_df = all_eps.merge(first_booking, on=["episode_id", "run_name"], how="left")
momentum_df["never_booked"] = momentum_df["first_booking_step"].isna()

print("Fraction that never produced a booking:")
print(momentum_df.groupby("run_name")["never_booked"].mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of first booking step (episodes that do book)
for rn in run_names:
    sub = momentum_df[(momentum_df["run_name"] == rn) & (~momentum_df["never_booked"])]["first_booking_step"]
    axes[0].hist(sub, bins=20, alpha=0.5, label=rn, color=RUN_PALETTE[rn], density=True)
axes[0].set_xlabel("Step index of first booking")
axes[0].set_ylabel("Density")
axes[0].set_title("3.5a Time-to-First-Booking Distribution\n(episodes that book)")
axes[0].legend(title="Run")

# Cumulative fraction of episodes with ≥1 booking by step position
max_step_global = int(steps_df["step_index"].max())
step_range = range(0, max_step_global + 1)
for rn in run_names:
    rn_eps = momentum_df[momentum_df["run_name"] == rn]
    n_total = len(rn_eps)
    cum = []
    for s in step_range:
        booked = (rn_eps["first_booking_step"].fillna(np.inf) <= s).sum()
        cum.append(booked / n_total)
    axes[1].plot(list(step_range), cum, label=rn, color=RUN_PALETTE[rn], linewidth=1.8)

axes[1].set_xlabel("Step index")
axes[1].set_ylabel("Fraction of episodes with ≥1 booking")
axes[1].set_title("3.5b Cumulative Booking Onset by Step")
axes[1].legend(title="Run")

fig.suptitle("3.5 Planning Momentum: When Booking Begins")
plt.tight_layout()
plt.show()

### 3.6 Itinerary Accumulation Over Steps

How does the number of booked items grow across the episode? We track whether `itinerary_snapshot` is non-null at each step (1 = has booking, 0 = no booking yet), then average across episodes to see the accumulation rate. A run that front-loads bookings demonstrates more decisive early planning.

In [ ]:
# Average has_booking flag by step_index and run_name (up to step 30 for legibility)
MAX_STEP_PLOT = 30

accum_df = (
    steps_df[steps_df["step_index"] <= MAX_STEP_PLOT]
    .groupby(["run_name", "step_index"])["has_booking"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
for rn in run_names:
    sub = accum_df[accum_df["run_name"] == rn]
    ax.plot(sub["step_index"], sub["has_booking"],
            label=rn, color=RUN_PALETTE[rn], linewidth=2, marker="o", markersize=3)

ax.set_xlabel("Step index")
ax.set_ylabel("Fraction of episodes with ≥1 booking at this step")
ax.set_title(f"3.6 Booking Presence Rate Over Steps (steps 0–{MAX_STEP_PLOT})")
ax.legend(title="Run")
plt.tight_layout()
plt.show()

### 3.7 Tool Diversity (Shannon Entropy) Over Relative Episode Progress

Entropy measures how spread out the tool choices are. **High entropy** early in an episode means the agent is exploring (calling many different tools). **Low entropy** in the middle or late stages means it has locked into a repetitive routine. A run that maintains diversity throughout may be more adaptive; one that collapses to low entropy early may be stuck.

We compute entropy at each relative decile (0–10%, 10–20%, … 90–100% of episode progress).

In [ ]:
def shannon_entropy(counts: np.ndarray) -> float:
    total = counts.sum()
    if total == 0:
        return 0.0
    probs = counts / total
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

entropy_rows = []
for rn in run_names:
    sub = steps_df[steps_df["run_name"] == rn]
    for decile in range(10):
        decile_steps = sub[sub["rel_decile"] == decile]["tool_name"]
        counts = np.array(list(Counter(decile_steps).values()), dtype=float)
        entropy_rows.append({
            "run_name": rn,
            "decile": decile,
            "rel_pct": decile * 10 + 5,   # midpoint label
            "entropy": shannon_entropy(counts),
            "n_steps": len(decile_steps),
        })

entropy_df = pd.DataFrame(entropy_rows)

fig, ax = plt.subplots(figsize=(10, 4))
for rn in run_names:
    sub = entropy_df[entropy_df["run_name"] == rn]
    ax.plot(sub["rel_pct"], sub["entropy"],
            label=rn, color=RUN_PALETTE[rn], linewidth=2, marker="s", markersize=5)

ax.set_xlabel("Relative episode progress (%)")
ax.set_ylabel("Shannon entropy H (bits)")
ax.set_title("3.7 Tool Diversity Over Episode Progress (per decile)")
ax.set_xticks(range(5, 100, 10))
ax.set_xticklabels([f"{d*10}–{d*10+10}%" for d in range(10)], rotation=30, ha="right")
ax.legend(title="Run")
plt.tight_layout()
plt.show()

print("\nMean entropy by run:")
print(entropy_df.groupby("run_name")["entropy"].agg(["mean", "std"]).round(3))

### 3.8 Redundancy: Consecutive & Near-Consecutive Repetitions

A **consecutive repeat** is when `tool[i] == tool[i+1]` — the agent calls the same tool back-to-back. This is a strong signal of a thought-repetition loop. A **window repeat** is a looser notion: the same tool appearing within a window of ±2 steps.

High redundancy correlates with poor planning efficiency and the tool-repetition behaviour flagged in the baseline episode analysis (notebook 11).

In [ ]:
WINDOW = 2  # steps on each side

redundancy_rows = []
for rn in run_names:
    sub = steps_df[steps_df["run_name"] == rn].sort_values(["episode_id", "step_index"])
    consec_repeats = 0
    window_repeats = 0
    total_steps = 0
    consec_pairs = 0
    highly_redundant_eps = 0
    n_eps = 0

    for ep_id, grp in sub.groupby("episode_id"):
        tools = grp["tool_name"].tolist()
        n = len(tools)
        n_eps += 1
        ep_consec = sum(1 for i in range(n - 1) if tools[i] == tools[i + 1])
        consec_repeats += ep_consec
        consec_pairs   += max(n - 1, 0)

        ep_window = sum(
            1 for i in range(n)
            if tools[i] in (
                tools[max(0, i - WINDOW):i] + tools[i + 1:i + WINDOW + 1]
            )
        )
        window_repeats += ep_window
        total_steps    += n
        if n > 1 and ep_consec / (n - 1) > 0.30:
            highly_redundant_eps += 1

    redundancy_rows.append({
        "run_name": rn,
        "consecutive_repeat_rate": consec_repeats / max(consec_pairs, 1),
        "window_repeat_rate":      window_repeats / max(total_steps, 1),
        "highly_redundant_ep_pct": highly_redundant_eps / max(n_eps, 1),
    })

redundancy_df = pd.DataFrame(redundancy_rows).set_index("run_name")
print(redundancy_df.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ["consecutive_repeat_rate", "window_repeat_rate", "highly_redundant_ep_pct"]
titles  = ["Consecutive repeat rate\n(tool[i]==tool[i+1])",
           f"Window repeat rate\n(same tool within ±{WINDOW} steps)",
           "Fraction of episodes\n>30% consecutive repeats"]

for ax, metric, title in zip(axes, metrics, titles):
    vals = redundancy_df[metric]
    bars = ax.bar(vals.index, vals.values,
                  color=[RUN_PALETTE[rn] for rn in vals.index],
                  edgecolor="white", alpha=0.85)
    ax.set_ylim(0, min(1.05, vals.max() * 1.3 + 0.05))
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis="x", rotation=20)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

fig.suptitle("3.8 Tool-Use Redundancy by Run")
plt.tight_layout()
plt.show()

---
## 4. Summary Insights Table

One row per run_name. Aggregates the key sequential behaviour metrics derived above.

In [ ]:
summary_rows = []
for rn in run_names:
    rn_steps = steps_df[steps_df["run_name"] == rn]
    rn_itin  = itin_df[itin_df["run_name"] == rn]
    rn_mom   = momentum_df[momentum_df["run_name"] == rn]
    rn_redund = redundancy_df.loc[rn] if rn in redundancy_df.index else {}
    rn_cascade = cascade_df.loc[rn] if rn in cascade_df.index else {}
    rn_ent   = entropy_df[entropy_df["run_name"] == rn]["entropy"].mean()

    ep_lens = rn_steps.groupby("episode_id")["step_index"].max()
    top_bigram = top_ngrams(steps_df, rn, 2, k=1)
    top_bigram_str = top_bigram.index[0] if len(top_bigram) else "—"

    summary_rows.append({
        "run_name":              rn,
        "n_episodes":            rn_steps["episode_id"].nunique(),
        "avg_seq_len":           round(ep_lens.mean(), 1),
        "n_unique_tools":        rn_steps["tool_name"].nunique(),
        "most_common_tool":      rn_steps["tool_name"].value_counts().index[0],
        "pct_with_itinerary":   round(rn_itin["has_itinerary"].mean() * 100, 1),
        "median_ttfb":           round(rn_mom["first_booking_step"].median(), 1),
        "pct_never_booked":     round(rn_mom["never_booked"].mean() * 100, 1),
        "error_retry_rate":     round(rn_cascade.get("retry_rate", float("nan")), 3),
        "consec_repeat_rate":   round(rn_redund.get("consecutive_repeat_rate", float("nan")), 3),
        "mean_entropy_bits":    round(rn_ent, 3),
        "top_bigram":            top_bigram_str,
    })

summary = pd.DataFrame(summary_rows).set_index("run_name")
print("\n=== SUMMARY INSIGHTS TABLE ===")
display(summary)

In [ ]:
# Radar / spider chart — normalise each metric to [0, 1] and overlay runs
radar_cols = [
    "avg_seq_len", "pct_with_itinerary", "median_ttfb",
    "error_retry_rate", "consec_repeat_rate", "mean_entropy_bits",
]
radar_labels = [
    "Avg steps", "% with\nitinerary", "Steps-to-\nfirst-booking",
    "Error\nretry rate", "Consec\nrepeat rate", "Tool\nentropy",
]

radar_data = summary[radar_cols].copy().astype(float)
# Normalise: invert metrics where lower is better (median_ttfb, consec_repeat, retry_rate)
for c in ["median_ttfb", "error_retry_rate", "consec_repeat_rate"]:
    if c in radar_data.columns:
        radar_data[c] = radar_data[c].max() - radar_data[c]   # invert so higher = better

col_max = radar_data.max()
col_max[col_max == 0] = 1.0
radar_norm = radar_data / col_max

N = len(radar_cols)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for rn in run_names:
    if rn not in radar_norm.index:
        continue
    values = radar_norm.loc[rn].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=rn, color=RUN_PALETTE[rn])
    ax.fill(angles, values, alpha=0.08, color=RUN_PALETTE[rn])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=8)
ax.set_yticklabels([])
ax.set_title("4. Behavioural Profile Radar\n(higher = better on all axes after inversion)",
             pad=20, fontsize=11)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), title="Run")
plt.tight_layout()
plt.show()